# NYC Green Taxi – MLOps Project Walkthrough

This notebook walks through the whole project, step by step. It **uses the real project code in `src/`**, so everything shown here is exactly what the pipeline, the API and the CI run.

| Step | What | Tool |
|---|---|---|
| 1 | Reusable training code | Python modules in `src/` |
| 2 | Versioned data and model | DVC |
| 3 | Experiment tracking and model registry | MLflow |
| 4 | Automated retraining | GitHub Actions |
| 5 | Prediction API | FastAPI |
| 6 | System and model monitoring | Prometheus, Grafana |
| 7 | Data drift and retraining trigger | Evidently |

**Before running:** open this notebook from the main project folder and run `uv run dvc repro` once, so the data, the model and the MLflow database exist.

In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    raise RuntimeError("Open this notebook from the main project folder (the one that contains src/).")

# Make the project modules importable, exactly like `uv run python src/...` does.
sys.path.insert(0, str(ROOT / "src"))
print("Project folder:", ROOT)

## Step 1 – Training code outside the notebook

The original notebook logic is split into three small modules:

- `prepare_data.py` downloads the January 2024 green taxi data.
- `preprocess.py` computes the ride duration and removes outliers.
- `train.py` trains, evaluates, saves the model and logs it to MLflow.

Here we import and use them directly.

In [ ]:
from prepare_data import DATA_PATH, download_data
from preprocess import FEATURES, TARGET, preprocess

download_data()
raw = pd.read_parquet(DATA_PATH)
clean = preprocess(raw)

print(f"Raw trips:   {len(raw):,}")
print(f"Clean trips: {len(clean):,}  (duration 1–60 min, distance 0.1–100 miles)")
clean.head()

**Model scope:** `trip_distance` is the taximeter distance, which is only known *after* a trip ends. The model replays completed trips; it is not a pickup-time estimate.

In [ ]:
clean[["trip_distance", TARGET]].describe().round(2)

Now we train with the same function the pipeline uses. This cell does **not** save anything or log to MLflow, so it does not change your project.

In [ ]:
from train import train

pipeline, rmse, n_train, n_test = train(raw)
print(f"Test RMSE: {rmse:.2f} minutes  (train={n_train:,}, test={n_test:,})")

## Step 2 – Data and model versioned with DVC

`dvc.yaml` defines two stages: `prepare_data` → `train_model`. `dvc.lock` stores a fingerprint (md5 hash) of every input and output, so any result can be reproduced exactly.

In [ ]:
print((ROOT / "dvc.yaml").read_text())

In [ ]:
def run(*args):
    result = subprocess.run([sys.executable, "-m", *args], capture_output=True, text=True, cwd=ROOT)
    print(result.stdout or result.stderr)

run("dvc", "status")
run("dvc", "metrics", "show")

## Step 3 – Experiment tracking with MLflow

Every training run logs its parameters, the RMSE and the model. The model is registered as `green-taxi-ride-duration` with the alias **production**.

In [ ]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlflow.db'}")

if not (ROOT / "mlflow.db").exists():
    print("No mlflow.db yet. Run `uv run dvc repro` first.")
else:
    runs = mlflow.search_runs(experiment_names=["green-taxi-monitoring"])
    columns = ["start_time", "metrics.rmse", "tags.developer", "params.month"]
    display(runs[[c for c in columns if c in runs.columns]].head(10))

    production = MlflowClient().get_model_version_by_alias("green-taxi-ride-duration", "production")
    print(f"Production model: version {production.version}")

## Step 4 – Automated retraining with GitHub Actions

`.github/workflows/train.yaml` runs on every push to the code, on the **Run workflow** button, and every Monday. It:

1. installs the locked dependencies (`uv sync --locked`),
2. runs the DVC pipeline (`dvc repro`) – download, train, log to MLflow,
3. shows the metrics,
4. smoke-tests the API,
5. uploads the model, metrics and MLflow run as an artifact.

In [ ]:
print((ROOT / ".github" / "workflows" / "train.yaml").read_text())

## Step 5 – Serving the model with FastAPI

`src/api.py` loads the model on startup and offers `/health`, `/predict` and `/metrics`. We use FastAPI's test client, so no server is needed. The request log is redirected to a temporary file, so the real monitoring data stays untouched.

In [ ]:
from fastapi.testclient import TestClient
import api

api.REQUEST_LOG_PATH = Path(tempfile.mkdtemp()) / "requests.csv"
client = TestClient(api.app)
client.__enter__()  # starts the API and loads the model

print("Health:", client.get("/health").json())

trip = {"PULocationID": 74, "DOLocationID": 75, "trip_distance": 1.5}
print("Prediction:", client.post("/predict", json=trip).json())

invalid = client.post("/predict", json={**trip, "trip_distance": -1})
print("Invalid input ->", invalid.status_code, "(rejected by the input schema)")

## Step 6 – Monitoring with Prometheus and Grafana

The API exposes metrics at `/metrics`. Prometheus collects them every 5 seconds, and Grafana shows them on the **Green Taxi API Monitoring** dashboard.

- **System behaviour:** number of requests, errors, latency.
- **Model behaviour:** histogram of predicted durations.

Let's send a few real trips and look at the metrics.

In [ ]:
for t in clean[FEATURES].sample(50, random_state=1).to_dict(orient="records"):
    client.post("/predict", json={
        "PULocationID": int(t["PULocationID"]),
        "DOLocationID": int(t["DOLocationID"]),
        "trip_distance": float(t["trip_distance"]),
    })

metrics_text = client.get("/metrics").text
for line in metrics_text.splitlines():
    if line.startswith(("api_prediction_requests_total", "api_prediction_errors_total",
                        "api_predicted_duration_minutes_count", "api_predicted_duration_minutes_sum")):
        print(line)

In the live setup (`docker compose up --build -d`), the same numbers appear in Grafana at http://127.0.0.1:3000.

## Step 7 – Data drift with Evidently

We compare the training data (reference) with two batches of 200 trips:

- **normal:** real trips from the same data,
- **shifted:** the same trips with `trip_distance` × 3, to simulate a change in the real world.

In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset
from generate_report import COLUMNS, load_reference
from preprocess import to_records

model = joblib.load(ROOT / "models" / "model.joblib")
reference = load_reference()

def with_predictions(df):
    df = df.copy()
    df["prediction"] = model.predict(to_records(df[FEATURES]))
    return df

sample = clean[FEATURES].sample(200, random_state=7)
shifted_sample = sample.copy()
shifted_sample["trip_distance"] = (shifted_sample["trip_distance"] * 3).clip(upper=100)

batches = {"normal": with_predictions(sample), "shifted": with_predictions(shifted_sample)}
snapshots = {
    name: Report(metrics=[DataDriftPreset()]).run(reference_data=reference[COLUMNS], current_data=batch[COLUMNS])
    for name, batch in batches.items()
}

In [ ]:
def drift_scores(snapshot):
    scores = {}
    for metric in snapshot.dict()["metrics"]:
        column = metric.get("config", {}).get("column")
        if column:
            scores[column] = round(float(metric["value"]), 3)
    return scores

pd.DataFrame({name: drift_scores(s) for name, s in snapshots.items()})

**How to read this:** a higher score means the batch looks more different from the training data. In the shifted batch, `trip_distance` drifts strongly, and the **predictions drift too**, because the model reacts to its inputs. The location IDs are categories; with only 200 requests spread over many taxi zones, their test is sensitive to small samples.

The full Evidently report for the shifted batch:

In [ ]:
snapshots["shifted"]

## Step 8 – From drift to retraining

`src/check_drift.py` turns the report into a decision. It retrains only on **meaningful** drift: at least 100 requests and a score above the trigger threshold in `trip_distance` or in the predictions.

In [ ]:
from check_drift import MIN_REQUESTS, TRIGGER_COLUMNS, TRIGGER_THRESHOLD

for name, snapshot in snapshots.items():
    scores = drift_scores(snapshot)
    drifted = [c for c in TRIGGER_COLUMNS if scores[c] > TRIGGER_THRESHOLD]
    enough = len(batches[name]) >= MIN_REQUESTS
    decision = "RETRAIN" if enough and drifted else "no action"
    print(f"{name:8s} -> {decision:9s}  drifted trigger columns: {drifted or 'none'}")

print(f"\nRule: >= {MIN_REQUESTS} requests and score > {TRIGGER_THRESHOLD} in {TRIGGER_COLUMNS}")
print("In the project: uv run python src/check_drift.py --retrain")

In [ ]:
client.__exit__(None, None, None)  # stop the test API

## Summary

| Requirement | Done with |
|---|---|
| Training code reusable outside a notebook | `src/prepare_data.py`, `preprocess.py`, `train.py` |
| Training data versioned with DVC | `dvc.yaml`, `dvc.lock` |
| Runs and artifacts tracked with MLflow | runs, metrics, registry alias `production` |
| Retraining automated with GitHub Actions | `.github/workflows/train.yaml` |
| Model served through an API | `src/api.py` (FastAPI) |
| Monitoring with Evidently, Prometheus, Grafana | `/metrics`, Grafana dashboard, drift reports |
| Bonus: baseline vs shifted batch | Step 7 |
| Bonus: drift-triggered retraining | `src/check_drift.py` |
| Bonus: smoke test | `src/check_api.py`, runs in CI |

**Next steps:** store the data in a DVC remote, pull new data before retraining, and deploy the stack to the cloud with Terraform.